In [1]:
# import libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import glob

In [6]:
# load all the data
wishlist = pd.read_csv('Books - Wishlist.csv')
wishlist.shape

(29, 7)

In [11]:
# load all yearly files into a list
yearly_files = ['Books - 2023.csv', 'Books - 2024.csv', 'Books - 2025.csv', 'Books - 2026.csv']
len(yearly_files)

4

In [12]:
# define a function to clean and standardize the data of each yearly file
def clean_yearly_data(file_path):
    df = pd.read_csv(file_path)
    df = df.rename(columns={'Book title': 'Name'}) # so it matches the wishlist column name
    df['Vote_Score'] = df['Vote'].str.count('⭐').fillna(0).astype(int) # convert the votes in numbers
    return df[['Name', 'Author', 'Genre', 'Vote_Score']] # we will ony need this columns for the model

In [13]:
# process and combine all yearly files
read_books_list = [clean_yearly_data(f) for f in yearly_files]
all_read_books = pd.concat(read_books_list, ignore_index=True)

In [14]:
# clean wishlist data
wishlist = wishlist[['Name', 'Author', 'Genre']]
wishlist['Author'] = wishlist['Author'].fillna('Unknown')
wishlist['Genre'] = wishlist['Genre'].fillna('Unknown')

print(f"Total read books processed: {len(all_read_books)}")
print(f"Total wishlist books processed: {len(wishlist)}")

Total read books processed: 18
Total wishlist books processed: 29


**Creating the "Metadata Soup"**
A machine learning model can't "read" a book the way we do. We need to combine the important text features into a single string (the "soup") that represents the essence of the book.

Since your project includes foreign titles (Korean/Japanese), combining the Author and Genre is our safest bet. This ensures that even if the titles are in different scripts, the model connects books by the same author or within the same category.

The Plan:
- Normalize Text: Convert everything to lowercase and remove spaces between words (so "Sci Fi" becomes "scifi"). This prevents the model from thinking "Sci" and "Fi" are two different things.
- Combine Columns: Merge Author and Genre into a new column called soup.

In [15]:
# normalize the text and combine merge author + genre
def create_soup(x):
    author = str(x['Author']).lower().replace(" ", "")
    genre = str(x['Genre']).lower().replace(" ", "")
    return f"{author} {genre}"

# apply this function to the wishlist and read books
all_read_books['soup'] = all_read_books.apply(create_soup, axis=1)
wishlist['soup'] = wishlist.apply(create_soup, axis=1)

# check it
print("Sample 'Soup' for Wishlist:")
print(wishlist[['Name', 'soup']].head())

Sample 'Soup' for Wishlist:
                                            Name  \
0  Записки изъ подполья (Notes from Underground)   
1                                Brave New World   
2                                 Fahrenheit 451   
3                           A Wizard of Earthsea   
4    Women, Race & Class (Donne, razza e colore)   

                                                soup  
0  фёдормихайловичдостоевский(fyodordostoevsky) e...  
1                      aldoushuxley dystopian,satire  
2                 raybradbury dystopian,socialsci-fi  
3                         ursulak.leguin highfantasy  
4               angeladavis feministtheory,sociology  


**Why "Soup"?**
Think of it like a cooking metaphor:
- Ingredients: Your raw data (Author, Genre, Language, Year).
- The Pot: The single string/column we are creating.
- The Result: A blended "soup" where all the individual flavors (features) are mixed together.

By the time the machine learning model "tastes" the data, it doesn't see "Author" or "Genre" as separate columns anymore. It just sees one big string of text. 

Using the term **Metadata Soup** (popularized in many Kaggle competitions and tutorials) simply helps programmers remember that we are blending multiple distinct categories into one unified text feature.

**Vectorization** (Turning Text into Math)
Computers are great at math but terrible at understanding "Fantasy" or "Mystery." We need to **convert our soup into numbers**. We will use a technique called **TF-IDF (Term Frequency-Inverse Document Frequency)**.

- **Term Frequency**: How many times does "Fantasy" appear in this book's soup?

- **Inverse Document Frequency**: How rare is "Fantasy" across all your books? If every single book you own is Fantasy, that word isn't very helpful for distinguishing between them. TF-IDF will give more weight to rarer, more specific words.

In [16]:
# 'stop_words' removes common words like 'the', 'and', etc.
tfidf = TfidfVectorizer(stop_words='english')

# combine all soups to make sure the model knows all possible words
combined_data = pd.concat([all_read_books['soup'], wishlist['soup']])

# transform the text data into a TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(combined_data)

print(f"Matrix shape: {tfidf_matrix.shape}")
print("Your books are now represented as numbers!")

Matrix shape: (47, 100)
Your books are now represented as numbers!


The shape (47, 100) tells us that across your 47 books, the model has identified 100 unique "descriptors" (specific authors and genre terms) to use as coordinates.

Imagine each book is now a point in a 100-dimensional space. Books by the same author or in the same genre will be clustered close together in that space

**Calculating Similarity**\
Now we use **cosine similarity**. This math calculation looks at the angle between the vectors (the rows of numbers) of two books. 
- If the angle is $0^\circ$, the books are identical (Similarity = $1.0$).
- If the angle is $90^\circ$, they share nothing (Similarity = $0.0$).

In [17]:
# calculate the similarity between every book and every other book
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# create a mapping of book titles to their index in the matrix
indices = pd.Series(
    pd.concat([all_read_books['Name'], wishlist['Name']]).index, 
    index=pd.concat([all_read_books['Name'], wishlist['Name']])
).drop_duplicates()

print("Similarity matrix calculated successfully!")

Similarity matrix calculated successfully!


In [19]:
# define a function to get recommendations 
def recommend_from_wishlist(book_title, wishlist_df, all_books_df, sim_matrix):
    if book_title not in indices:
        return "Book not found in your records."
    
    idx = indices[book_title]
    sim_scores = list(enumerate(sim_matrix[idx])) # get the similarity scores for the given book
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) # sort by score, highest first
    sim_scores = sim_scores[1:10] # get the scores of the most similar books
    book_indices = [i[0] for i in sim_scores] # books indices

    # return only the books that are in the wishlist
    results = []
    for i in book_indices:
        book_name = pd.concat([all_read_books, wishlist])['Name'].iloc[i]
        if book_name in wishlist_df['Name'].values:
            results.append(book_name)
            
    return results[:5] # top 5 matches from wishlist

# --- TEST IT OUT ---
my_recommendations = recommend_from_wishlist('Project Hail Mary', wishlist, all_read_books, cosine_sim)
print(f"If you liked that, you might want to read these from your wishlist:\n{my_recommendations}")

If you liked that, you might want to read these from your wishlist:
['Artemis', 'The Three-Body Problem Trilogy', 'Fahrenheit 451', 'This Is How You Lose the Time War']


In [ ]:
import requests # to connect to the Google Books API

# create a function to get external recommendations based on favorite genres/authors
def get_external_recommendations(read_df, wishlist_df):
    favorites = read_df[read_df['Vote_Score'] >= 4]
    if favorites.empty:
        return "No high-rated books found to base recommendations on."

    # create an unique list of genres and authors you liked
    fav_genres = favorites['Genre'].unique()
    fav_authors = favorites['Author'].unique()

    # list to store new suggestions from the API
    new_suggestions = []
    
    # track the titles that we already know about to avoid duplicates
    known_titles = set(read_df['Name'].str.lower().tolist() + 
                       wishlist_df['Name'].str.lower().tolist())
    print(f"Searching for new books based on your love for {fav_genres[0]}...")

    # use the Google Books API
    query = f"subject:{fav_genres[0]}"
    url = f"https://www.googleapis.com/books/v1/volumes?q={query}&orderBy=relevance&maxResults=10"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        for item in data.get('items', []):
            info = item.get('volumeInfo', {})
            title = info.get('title')
            authors = info.get('authors', ['Unknown Author'])
            
            if title.lower() not in known_titles: # to add new books
                new_suggestions.append({
                    'Title': title,
                    'Author': ", ".join(authors),
                    'Description': info.get('description', 'No description available.')[:150] + "..."
                })
                
    except Exception as e:
        return f"Error connecting to API: {e}"

    return pd.DataFrame(new_suggestions).head(5)

# --- EXECUTION ---
new_discoveries = get_external_recommendations(all_read_books, wishlist)

print("\n--- NEW BOOK DISCOVERIES ---")
print(new_discoveries[['Title', 'Author']])

Searching for new books based on your love for Horror, True Crime...

--- NEW BOOK DISCOVERIES ---
                                      Title          Author
0                                    Horror    Darryl Jones
1                           Horror Unmasked   Brad Weismann
2               Sleeping with the Lights on    Darryl Jones
3  Horror and Evil in the Name of Enjoyment    Han-yu Huang
4                            English Gothic  Jonathan Rigby
